# Install Libraries

In [ ]:
!pip install fastapi nest-asyncio uvicorn
!pip install -U transformers==4.39.3 huggingface_hub==0.20.3 tokenizers==0.15.2 sentencepiece -q

# Create Config File

In [ ]:
yaml_config =  """
               model_path: "VietAI/envit5-translation"
               """
with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_config)

In [ ]:
import torch
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "VietAI/envit5-translation"

tokenizer = T5Tokenizer.from_pretrained(
    model_name,
    use_fast=False,
    legacy=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

texts = [
  "vi: VietAI là tổ chức phi lợi nhuận với sứ mệnh ươm mầm tài năng về trí tuệ nhân tạo và xây dựng một cộng đồng các chuyên gia trong lĩnh vực trí tuệ nhân tạo đẳng cấp quốc tế tại Việt Nam.",
    "vi: Theo báo cáo mới nhất của Linkedin về danh sách việc làm triển vọng với mức lương hấp dẫn năm 2020, các chức danh công việc liên quan đến AI như Chuyên gia AI (Artificial Intelligence Specialist), Kỹ sư ML (Machine Learning Engineer) đều xếp thứ hạng cao.",
    "en: Our teams aspire to make discoveries that impact everyone, and core to our approach is sharing our research and tools to fuel progress in the field.",
    "en: We're on a journey to advance and democratize artificial intelligence through open source and open science."
]

tokenized = tokenizer(
    texts,
    return_tensors="pt",
    padding=True
)

tokenized = {k: v.to(device) for k, v in tokenized.items()}

outputs = model.generate(**tokenized, max_length=512)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

# Build Model

In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
from omegaconf import OmegaConf

class EnViT5Translation:
    def __init__(self, config_path):
        self.config = OmegaConf.load(config_path)
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.config.model_path,
            use_fast=False,
            legacy=False
        )

        self.model = T5ForConditionalGeneration.from_pretrained(self.config.model_path)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def __call__(self, text):
        input_ids = self.tokenizer(text, return_tensors="pt").input_ids.to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(input_ids, max_length=512)

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

# Initialize Model

In [ ]:
translator = EnViT5Translation("./config.yaml")

# Initialize API

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
import threading
import uvicorn
import time

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/translate")
async def translate(text: str):
    result = translator(text)
    return {
        "original_text": text,
        "translated_text": result
    }

def run_api():
    config = uvicorn.Config(
        app,
        host="127.0.0.1",
        port=8000,
        log_level="info"
    )
    server = uvicorn.Server(config)
    server.run()

thread = threading.Thread(target=run_api, daemon=True)
thread.start()
time.sleep(3)

# Call Local API

In [ ]:
import requests

API_URL = "http://127.0.0.1:8000/translate"
params = {"text": "en: Hello, my dog is cute"}

response = requests.get(API_URL, params=params)
print(response.json())


In [ ]:
# ssh -p 443 -R0:localhost:8000 a.pinggy.io